# Setting I/O Locations

# Importing Date

In [1]:
!!pip install openpyxl

StatementMeta(, 78aa149f-3942-40eb-8899-571a3897e817, 3, Finished, Available, Finished, False)

['Requirement already satisfied: openpyxl in /home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages (3.0.10)',
 'Requirement already satisfied: et_xmlfile in /home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages (from openpyxl) (1.1.0)']

In [2]:
files = spark._jvm.org.apache.hadoop.fs.FileSystem \
    .get(spark._jsc.hadoopConfiguration()) \
    .listStatus(spark._jvm.org.apache.hadoop.fs.Path("Files/CASH"))

file_names = [file.getPath().getName() for file in files]

print(file_names)



StatementMeta(, 78aa149f-3942-40eb-8899-571a3897e817, 4, Finished, Available, Finished, False)

['A.xlsx', 'B.xlsx', 'C.xlsx', 'D.xlsx', 'E.xlsx', 'F.xlsx', 'G.xlsx', 'H.xlsx', 'I.xlsx', 'J.xlsx', 'K.xlsx', 'L.xlsx', 'M.xlsx', 'N.xlsx']


In [3]:
import pandas as pd

df_list = []

for file in file_names:
    
    file_path = f"Files/CASH/{file}"
    print(f"Reading: {file_path}")
    
    # Read Excel using Spark
    df_spark = spark.read.format("excel") \
        .option("header", True) \
        .option("inferSchema", True) \
        .load(file_path)
    
    # Convert to pandas
    df_pd = df_spark.toPandas()
    
    df_list.append(df_pd)

# Combine all files
df = pd.concat(df_list, ignore_index=True)




StatementMeta(, 78aa149f-3942-40eb-8899-571a3897e817, 5, Finished, Available, Finished, False)

Reading: Files/CASH/A.xlsx
Reading: Files/CASH/B.xlsx
Reading: Files/CASH/C.xlsx
Reading: Files/CASH/D.xlsx
Reading: Files/CASH/E.xlsx
Reading: Files/CASH/F.xlsx
Reading: Files/CASH/G.xlsx
Reading: Files/CASH/H.xlsx
Reading: Files/CASH/I.xlsx
Reading: Files/CASH/J.xlsx
Reading: Files/CASH/K.xlsx
Reading: Files/CASH/L.xlsx
Reading: Files/CASH/M.xlsx
Reading: Files/CASH/N.xlsx


In [4]:
# Check rows where Selection Value 2 = 30000055
filtered_df = df[df["Selection Value 2"] == 30000055]



StatementMeta(, 78aa149f-3942-40eb-8899-571a3897e817, 6, Finished, Available, Finished, False)

In [5]:
# Rename the column
df.rename(columns={'Selection Value 2': 'Contract'}, inplace=True)

# Convert to numeric (coerce errors if any)
df['Contract'] = pd.to_numeric(df['Contract'], errors='coerce')

# Convert to whole number (integer)
df['Contract'] = df['Contract'].astype('Int64')  # nullable integer

# Convert to string
df['Contract'] = df['Contract'].astype(str)





StatementMeta(, 78aa149f-3942-40eb-8899-571a3897e817, 7, Finished, Available, Finished, False)

In [6]:
# Sum of Payment Amount column
total_payment = df["Payment Amount"].sum()




StatementMeta(, 78aa149f-3942-40eb-8899-571a3897e817, 8, Finished, Available, Finished, False)

In [7]:
df["Contract"] = pd.to_numeric(df["Contract"], errors="coerce")





StatementMeta(, 78aa149f-3942-40eb-8899-571a3897e817, 9, Finished, Available, Finished, False)

# Remove Duplicates

In [8]:
# Making a Key


# Ensure Payment Amount is numeric
df['Payment Amount'] = pd.to_numeric(df['Payment Amount'], errors='coerce')

# Ensure Value Date is string (format can be customized)
df['Value Date'] = pd.to_datetime(df['Value date'], errors='coerce').dt.strftime('%Y-%m-%d')

# Create composite key
df['Key'] = df['Contract'].astype(str) + "_" + df['Value Date'] + "_" + df['Payment Amount'].astype(str)




StatementMeta(, 78aa149f-3942-40eb-8899-571a3897e817, 10, Finished, Available, Finished, False)

In [9]:
filtered_df = df[df["Contract"] == 30000055]



StatementMeta(, 78aa149f-3942-40eb-8899-571a3897e817, 11, Finished, Available, Finished, False)

In [10]:
# Keep only one row per unique Key
df = df.drop_duplicates(subset='Key', keep='first').reset_index(drop=True)



StatementMeta(, 78aa149f-3942-40eb-8899-571a3897e817, 12, Finished, Available, Finished, False)

In [12]:
# Sum of Payment Amount column
total_payment = df["Payment Amount"].sum()



StatementMeta(, 78aa149f-3942-40eb-8899-571a3897e817, 14, Finished, Available, Finished, False)

In [13]:
# Drop the Key column
df = df.drop(columns=['Key'])





StatementMeta(, 78aa149f-3942-40eb-8899-571a3897e817, 15, Finished, Available, Finished, False)

# Date Format Conversion

In [14]:
# Convert Value Date to datetime
df['Value Date'] = pd.to_datetime(df['Value Date'], errors='coerce')
df['Posting Date'] = pd.to_datetime(df['Posting Date'], errors='coerce')




StatementMeta(, 78aa149f-3942-40eb-8899-571a3897e817, 16, Finished, Available, Finished, False)

# Sorting by Date

In [16]:
# Sort dataframe by Value Date (earliest first)
df= df.sort_values(by='Value Date', ascending=True).reset_index(drop=True)




StatementMeta(, 78aa149f-3942-40eb-8899-571a3897e817, 17, Finished, Available, Finished, False)

# Paymnet No.

In [17]:
# Generate running payment number per Contract in df
df['Payment No.'] = df.groupby('Contract').cumcount() + 1

# Convert to integer type
df['Payment No.'] = df['Payment No.'].astype('Int64')




StatementMeta(, 78aa149f-3942-40eb-8899-571a3897e817, 18, Finished, Available, Finished, False)

In [18]:
# Sum of Payment Amount column
PaymentNo= df["Payment No."].max()



StatementMeta(, 78aa149f-3942-40eb-8899-571a3897e817, 19, Finished, Available, Finished, False)

In [19]:
filtered_df = df[df["Contract"] == 30000055]



StatementMeta(, 78aa149f-3942-40eb-8899-571a3897e817, 20, Finished, Available, Finished, False)

# Pivoted Payments

In [20]:
import pandas as pd

# Ensure Value Date is datetime
df['Value Date'] = pd.to_datetime(df['Value Date'], errors='coerce')

# Remove missing Payment No.
df = df[df['Payment No.'].notna()]

# Convert Payment No. to integer
df['Payment No.'] = df['Payment No.'].astype(int)

# Pivot to have Payment No. as columns with Value Date and Payment Amount
pivot = df.pivot(index='Contract', columns='Payment No.', values=['Value Date', 'Payment Amount'])

# Sort Payment No. columns
pivot = pivot.sort_index(axis=1, level=1)

# Flatten MultiIndex columns
pivot.columns = [f"{col} {pn}" for col, pn in pivot.columns]

# Add totals per contract
pivot['Total Amount'] = df.groupby('Contract')['Payment Amount'].sum()
pivot['Total Max of Value date'] = df.groupby('Contract')['Value Date'].max()
pivot['Max Payment No.'] = df.groupby('Contract')['Payment No.'].max()

# Reset index to make Contract a column
pivot = pivot.reset_index()

# Optional: format Value Date and Payment Amount for readability
for col in pivot.columns:
    if 'Value Date' in col:
        pivot[col] = pd.to_datetime(pivot[col], errors='coerce').dt.strftime('%m/%d/%Y')
    if 'Payment Amount' in col:
        pivot[col] = pivot[col].map(lambda x: f"{x:,.2f}" if pd.notnull(x) else '')



StatementMeta(, 78aa149f-3942-40eb-8899-571a3897e817, 21, Finished, Available, Finished, False)

# Output

In [22]:
# Define output folder path
output_folder = "/lakehouse/default/Files/Silver_Output/"

# Define full file path
file_path = output_folder + "Cash_Payment_Breakup.xlsx"

# Save file
pivot.to_excel(file_path, index=False)

print("Pivot exported successfully to:", file_path)


StatementMeta(, 78aa149f-3942-40eb-8899-571a3897e817, 23, Finished, Available, Finished, False)

Pivot exported successfully to: /lakehouse/default/Files/Silver_Output/Cash_Payment_Breakup.xlsx
